## Homework: Evaluation

In homework 2 we built keyword, vector, and hybrid search over the course
lessons, and ended with an open question: which one is best? The way to answer
that is to measure, and that's what we do here.

In this homework we generate a ground truth dataset and use it to evaluate
search, the same way we did in the module. There we only evaluated keyword
search. Here we also evaluate vector and hybrid search, so we can finally
compare them on numbers instead of intuition.

Like in homework 1 and 2, our knowledge base is the course lessons themselves.
Each module has a `lessons/` folder of numbered markdown pages, and we pull
them from GitHub. We use commit `8c1834d`, so everyone works with the exact
same 72 pages.

> It's possible your answers won't match exactly. If so, select the closest one.

## Setup

This homework continues from homework 2. We reuse the same chunks and the same
search functions, so it's easiest to keep working in the same project.

We need a few more libraries for generating questions with an LLM:

```bash
uv add openai pydantic python-dotenv pandas
```

For the LLM, we recommend OpenAI with `gpt-5.4-mini`, but you can use any model
and provider you like - just adapt the client accordingly. Put your key in a
`.env` file as in the earlier modules.

Load the data exactly as in homework 2:

```python
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
```

This gives 72 pages.

## Generating ground truth

To evaluate search, we need a dataset of questions where we know which document
is the correct answer. This is the ground truth.

We generate it the same way as in the module. For each lesson page, we ask an
LLM to write 5 questions that are answered by that page. Each question is then
labeled with the page it came from.

We use the same structured-output approach as in the module - the same
`Questions` model and the `llm_structured` helper from `evaluation_utils.py`.

Download `evaluation_utils.py` and the `rag_helper.py` it depends on:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
wget ${PREFIX}/04-evaluation/code/evaluation_utils.py
```

The module's instructions generate questions from a FAQ record, so we adapt
them for a lesson page:

```python
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()
```

We ask for different wording from the page on purpose. Real users don't phrase
their questions the way the lesson does, and copying the text would make the
evaluation too easy.

For each page, build a JSON user prompt from its `filename` and `content`, then
call `llm_structured` with the `Questions` model. Turn each returned question
into a record labeled with the page's `filename`. The call also returns the
token usage, the same as in the lessons.

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

Call in shell:
```
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
wget ${PREFIX}/04-evaluation/code/evaluation_utils.py
```

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

## Q1. Generating questions

Generating questions for all 72 pages costs money and takes time, so let's
start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`

Each call returns the token usage, which most LLM APIs report on the response
object (e.g. `response.usage.input_tokens` / `prompt_tokens`).

What's the average number of input tokens across these 3 calls?

* 140
* 1400
* 14000
* 140000

> These numbers vary between runs, even with the same model, so pick the closest
> option. A different provider or model may land further apart, but the input
> tokens stay in the same order of magnitude - the prompt we send is the same.

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
from evaluation_utils import llm_structured_retry
import json
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [8]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.00466875

In [10]:
input = 0
for u in usages:
    input += u.input_tokens
input/len(usages)

1353.0

## The full ground truth

You don't need to generate the data for the rest of the homework. We already
did it for all 72 pages, using the same approach as in the lessons, and saved
the 360 questions to a file.

Download it:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv
```

Load it with pandas into a list of records called `ground_truth`. Each record
has a `question` and the `filename` of the page that should answer it.

## Searching the chunks

We search over the same chunks as in homework 2.

Create them with `chunk_documents`:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

This gives 295 chunks.

Now rebuild the search from homework 2 over these chunks. Build a text index
(`Index`) and a vector index (`VectorSearch`), both keyed on `filename`. Wrap
each one in a function, `text_search` and `vector_search`, that takes a query
and the number of results to return (5 by default).

For hybrid search, reuse the `rrf` function from homework 2:

```python
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]
```

Then define `hybrid_search` on top of it:

```python
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)
```

In [11]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
from minsearch import Index

index = Index(
    text_fields=["content"]
)

index.fit(chunks)

def text_search(query, num_results=10):
    return index.search(query, num_results=num_results)

In [14]:
from embedder import Embedder
from tqdm.auto import tqdm
import numpy as np
from minsearch import VectorSearch


embed = Embedder()

texts = [doc["content"] for doc in chunks]

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)


vindex = VectorSearch()
vindex.fit(X, chunks)

def vector_search(query, num_results=10):
    query_vector = embed.encode(query)
    return vindex.search(query_vector, num_results=num_results)

  0%|          | 0/6 [00:00<?, ?it/s]

In [15]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [16]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search

Take the first question from the ground truth:

```python
q = ground_truth[0]["question"]
```

After running `text_search` for it, what's the `filename` of the first result?

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/03-rag.md`
* `01-agentic-rag/lessons/13-function-calling.md`
* `01-agentic-rag/lessons/10-rag-next-steps.md`

In [17]:
q = ground_truth[0]["question"]

In [18]:
text_search(q, 1)[0]['filename']

'01-agentic-rag/lessons/03-rag.md'

## Q3. First result with vector search

After running `vector_search` for the same question, what's the `filename` of
the first result?

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/03-rag.md`
* `04-evaluation/lessons/11-evaluation-intro.md`
* `04-evaluation/lessons/12-rag-answers.md`

This question was generated from `01-agentic-rag/lessons/01-intro.md`. Notice
that one method finds the right page at the top and the other doesn't. That's
exactly why we measure across the whole dataset instead of trusting one query.

In [19]:
vector_search(q, 1)[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

## Evaluation metrics

We evaluate search exactly as in the module, reusing the same functions from the
lecture. We change only the label. Our ground truth uses `filename`, so a result
counts as a hit when a returned chunk's `filename` matches the question's
`filename`, not a document `id`.

As a reminder, these functions do the following:

- `compute_relevance` runs search for a question and returns a list of 0s and 1s
- `hit_rate` is the fraction of questions where the correct page appears in the results
- `mrr` (Mean Reciprocal Rank) also rewards finding the page near the top
- `evaluate` runs a search function over the whole ground truth and returns both metrics

In [20]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [21]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

## Q4. Evaluating text search

Evaluate `text_search` on the ground truth data.

What's the Hit Rate?

* 0.55
* 0.66
* 0.76
* 0.88

In [22]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [23]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [24]:
hit_rate(relevance_total)

0.8416666666666667

## Q5. Evaluating vector search

Now evaluate `vector_search` - the part we left for the homework, since the
module only evaluated keyword search.

What's the MRR?

* 0.35
* 0.45
* 0.55
* 0.65

In [25]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [26]:
relevance_vector = compute_relevance_total(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [27]:
mrr(relevance_vector)

0.5646472663139328

## Q6. Tuning hybrid search

The `k` constant in RRF controls how much the top ranks matter. A smaller `k`
sharpens the gap between positions, so being at the top of a list counts for
more. The RRF paper uses 60 as a default, but the best value depends on the data
- so let's measure it.

Evaluate `hybrid_search` over the full ground truth dataset for `k` values 1,
50, 100, and 200. Compare the MRR values for these runs.

Which `k` gives the best MRR?

* 1
* 50
* 100
* 200

> Several values of `k` may give the same MRR. If there's a tie, pick the
> smallest `k`.

## Using this framework

You now have an `evaluate` function that takes any search function and returns
Hit Rate and MRR.

Use it to measure any change you make to search:

- tune the field boosts in keyword search
- try a different embedding model for vector search
- change `k` in the RRF formula for hybrid search
- change the number of results you return

Change a setting, re-run `evaluate`, and see whether the metric moves. The
ground truth stays fixed, so the comparison is fair. That's how you replace
guessing with measuring.

In [28]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [29]:
evaluate(
    ground_truth,
    hybrid_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}

In [34]:
def search_k(query, k):

    return hybrid_search(
        query,
        k=k,
    )

In [35]:
results = []
for k in [1, 50, 100, 200]:
    print(f"Evaluating k={k}.")
    result = evaluate(ground_truth,
        lambda query, k=k: search_k(
                    query,
                    k
                )


    )

    results.append({
        "k": k,
        "hit_rate": result["hit_rate"],
        "mrr": result["mrr"],
    })

Evaluating k=1.


  0%|          | 0/360 [00:00<?, ?it/s]

Evaluating k=50.


  0%|          | 0/360 [00:00<?, ?it/s]

Evaluating k=100.


  0%|          | 0/360 [00:00<?, ?it/s]

Evaluating k=200.


  0%|          | 0/360 [00:00<?, ?it/s]

In [36]:
results

[{'k': 1, 'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449},
 {'k': 50, 'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667},
 {'k': 100, 'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667},
 {'k': 200, 'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}]